## 1. Load the data and helpers
Loads the benchmark, the synthetic customer notes (with their true names), and the Enron emails (used to check for false alarms). Defines small helpers for reading the benchmark's labels, comparing spans, and showing tables.

In [1]:
import os, sys, re, json, html, random, importlib
import pandas as pd
from collections import Counter
from IPython.display import display, Markdown, HTML

DATA_DIR = os.path.expanduser("~/bank-rag-guard/data")
SRC_DIR = os.path.expanduser("~/bank-rag-guard/src")
EVAL_DIR = os.path.expanduser("~/bank-rag-guard/eval")
os.makedirs(SRC_DIR, exist_ok=True)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)                       # lets Python find our modules in src/

customers = pd.read_csv(f"{DATA_DIR}/customers.csv", dtype=str)    # includes the true "name" column
enron = pd.read_json(f"{DATA_DIR}/enron_emails.jsonl", lines=True, dtype=False, convert_dates=False)
bench = pd.read_json(f"{DATA_DIR}/pii_benchmark.jsonl", lines=True, dtype=False, convert_dates=False)

def get_mask(value):
    """privacy_mask is a list of {value, label, start, end}; it may be stored as a JSON string."""
    return json.loads(value) if isinstance(value, str) else value

def overlap(a_start, a_end, b_start, b_end):
    """Number of characters shared by two spans (0 if they do not touch)."""
    return max(0, min(a_end, b_end) - max(a_start, b_start))

def show(df, title, pct_cols=(), bars=None):
    """Display a DataFrame as a clean table. bars = {column: maximum value} draws in-cell bars."""
    display(Markdown(f"**{title}**"))
    styler = df.style.hide(axis="index").format({c: "{:.1%}" for c in pct_cols}, na_rep="–")
    for col, top in (bars or {}).items():
        styler = styler.bar(subset=[col], vmin=0, vmax=top, color="#5fa777")
    display(styler)

print(len(customers), "customer notes;", len(enron), "emails;", len(bench), "benchmark examples")

300 customer notes; 1500 emails; 1000 benchmark examples


## 2. How names appear in the benchmark
Shows (a) the words that most often come right before a labelled first name, which tells us which cue words are worth using, and (b) six real examples of each name-related label with surrounding text.

In [2]:
# --- (a) The word right before each first name ---
before_counts = Counter()
for row in bench.itertuples():
    text = row.source_text
    for g in get_mask(row.privacy_mask):
        if g["label"] == "FIRSTNAME":
            m = re.search(r"(\S+)\s*$", text[:g["start"]])        # last word before the name
            before_counts[m.group(1).lower() if m else "(start of text)"] += 1
show(pd.DataFrame(before_counts.most_common(20), columns=["Word before a first name", "Count"]),
     "Most common words before a first name", bars={"Count": before_counts.most_common(1)[0][1]})

# --- (b) Six examples per name label, with the name highlighted ---
shown = {l: 0 for l in ("FIRSTNAME", "LASTNAME", "MIDDLENAME", "PREFIX")}
body = ""
for row in bench.itertuples():
    text = row.source_text
    for g in get_mask(row.privacy_mask):
        if g["label"] in shown and shown[g["label"]] < 6:
            shown[g["label"]] += 1
            body += ("<tr><td><b>" + g["label"] + "</b></td>"
                     "<td style='font-family:monospace;text-align:left'>…" +
                     html.escape(text[max(0, g["start"] - 40):g["start"]]) +
                     "<mark style='background:#ffd54f;color:#000'>" + html.escape(g["value"]) + "</mark>" +
                     html.escape(text[g["end"]:g["end"] + 40]) + "…</td></tr>")
display(HTML("<table><tr><th>Label</th><th style='text-align:left'>Value in context</th></tr>" + body + "</table>"))

**Most common words before a first name**

Word before a first name,Count
(start of text),58
dear,48
hi,22
hello,21
for,13
with,10
hey,8
patient,5
i'm,4
mr.,4


Label,Value in context
FIRSTNAME,"…Dear Omer, as per our records, your license 78B5R…"
FIRSTNAME,…Kattie could you please share your recomndatio…
FIRSTNAME,"…Carleton, the new interactive educational tools …"
FIRSTNAME,…I'm Garrick Murray. I will be picking up the test r…
LASTNAME,…I'm Garrick Murray. I will be picking up the test results …
LASTNAME,…picking up the test results for patient Leuschke. We've been assigned number 21919415264…
FIRSTNAME,"…""Dear Carmen, As our National Brand Architect, we ne…"
FIRSTNAME,…Cardiac consult reminder for Stephen on 10/03/1976 at 4:40 AM. Commute to ou…
PREFIX,"…Ms.North, our office in Essex confirmed you…"
MIDDLENAME,"…Ms.North, our office in Essex confirmed your vis…"


## 3. Write the name finder (`src/person_names.py`)
Finds person names in English and Arabic using rules plus name lists, entirely on this server:
- **Name lists** come from the Faker library's built-in data (English and Arabic first and last names). No network is needed.
- **English cue rules** mask one to three capitalised words after a title (Mr, Dr, with or without a space after the dot), a greeting (Dear, Hi…), a sign-off (Regards, Thanks…), a role word (patient, customer, client…), or an introduction (I'm, my name is, named, called…).
- **Arabic cue rules** mask one to four Arabic words after a cue such as العميل, السيد, الدكتور, المريض or باسم, stopping at a stop word such as رقم (number) or at punctuation.
- **List rules:** two adjacent capitalised (English) or Arabic words are masked when the first is a known first name and the second is a known name.
- **First-name rules:** a single known first name is masked after context words such as for, with, contact and ask, and at the start of a sentence before a comma or a request word.
- **Stop-word lists** keep group and organisation words (Parents, Music Club, Legal, Customer Service) and months from being masked.
- Titles and cue words stay visible; only the name is replaced by `[NAME]`.
- **Wider name lists:** English names are combined from several English-speaking Faker locales (US, UK, Canada, Australia, Ireland, New Zealand, India).
- **Opening-name rule:** a capitalised word at the very start of a text or line, followed by a comma and a typical opening word (our, your, the, we, please…) or by a request word (could, can, would…), is masked as a name, unless it is a common sentence opener.
- **Arabic words** include their diacritics (such as the shadda mark), and Arabic names after a cue can run to six words, so names with bin / ibn / al- chains are masked in full.

In [3]:
%%writefile /home/linux1/bank-rag-guard/src/person_names.py
"""Bilingual (English/Arabic) person-name finder: cue words plus name lists. Runs entirely locally."""
import re

# ---------- Name lists, read from the Faker library's built-in data (no network needed) ----------
def _faker_names(locale, attribute):
    """Return one name list from a Faker locale as a set (empty if it cannot be read)."""
    try:
        module = __import__(f"faker.providers.person.{locale}", fromlist=["Provider"])
        return set(getattr(module.Provider, attribute, ()))
    except Exception:
        return set()

# English-script name lists combined from several Faker locales for wider coverage
EN_LOCALES = ["en_US", "en_GB", "en_CA", "en_AU", "en_IE", "en_NZ", "en_IN",
              "es_ES", "es_MX", "de_DE", "fr_FR", "it_IT", "pt_BR", "nl_NL"]
FIRST_EN, LAST_EN = set(), set()
for _loc in EN_LOCALES:
    FIRST_EN |= _faker_names(_loc, "first_names")
    LAST_EN |= _faker_names(_loc, "last_names")
FIRST_AR, LAST_AR = _faker_names("ar_AA", "first_names"), _faker_names("ar_AA", "last_names")

# ---------- Words that are capitalised but are not names (compared in lower case) ----------
NOT_NAMES = {
    "all", "everyone", "everybody", "team", "sir", "madam", "there", "colleagues", "customer", "customers",
    "service", "services", "support", "care", "department", "dept", "group", "inc", "corp", "corporation",
    "company", "llc", "ltd", "please", "thanks", "thank", "regards", "the", "this", "that", "these", "those",
    "hello", "hi", "dear", "mr", "mrs", "ms", "dr", "prof", "miss", "mx", "best", "kind", "sincerely", "yours",
    "cheers", "again", "regarding", "attached", "following", "monday", "tuesday", "wednesday", "thursday",
    "friday", "saturday", "sunday", "january", "february", "march", "april", "may", "june", "july", "august",
    "september", "october", "november", "december", "enron", "energy", "gas", "power", "bank", "hospital",
    "university", "school", "college", "center", "centre", "office", "manager", "director", "president",
    "officer", "assistant", "staff", "patient", "student", "client", "member", "user", "account", "online",
    # group and organisation words seen as false alarms after "Dear" / "Hi" / "Employee"
    "parent", "parents", "club", "squad", "legal", "medicine", "geriatric", "music", "society", "association",
    "committee", "foundation", "ministry", "council", "board", "friends", "family", "guardian",
    "guardians", "residents", "applicants", "students", "employees", "employee", "portal", "roll", "name",
    "sales", "marketing", "finance", "operations", "hr", "it", "admin", "administrator", "recipient",
    # common words and sentence openers that were wrongly masked as names
    "agent", "certification", "however", "therefore", "thus", "also", "yes", "no", "ok", "okay", "sure",
    "sorry", "unfortunately", "finally", "meanwhile", "otherwise", "hence", "furthermore", "moreover",
    "now", "then", "today", "tomorrow", "yesterday", "welcome", "congratulations", "attention", "notice",
    "reminder", "update", "urgent", "note", "warning", "first", "second", "next", "lastly", "overall",
    "indeed", "instead", "still", "here", "what", "why", "how", "when", "where", "who", "if", "so",
    "but", "and", "or", "because", "since", "although", "while", "as", "in", "on", "at", "of",
    "we", "hope", "greetings", "hey", "hell", "hiya", "i", "you", "our", "my", "they", "he", "she",
    "kindly", "good", "great", "happy", "let", "just", "meeting", "agreement",
}
AR_STOP = {
    "رقم", "الهوية", "الهاتف", "الجوال", "رقمه", "رقمها", "الذي", "التي", "في", "من", "على", "إلى", "الى", "عن",
    "هو", "هي", "قد", "قام", "قدم", "قدّم", "طلب", "أبلغ", "تقدم", "يطلب", "لديه", "لدى", "مع", "أن", "إن",
    "بخصوص", "بشأن", "حول", "الإماراتية", "المسجل", "المسجلة", "صاحب", "صاحبة", "الحساب", "البطاقة",
    "المصرفي", "المستخدم", "الشركة", "البنك", "الفرع", "وقد", "ولدينا", "كان", "كانت", "يرجى", "نود",
}

# ---------- English building blocks ----------
# A capitalised word: Smith, O'Brien, Smith-Jones, McGlynn
CAP = r"(?:Mc[A-Z][a-z]+|Mac[A-Z][a-z]+|[A-Z][a-z]*['\-][A-Z][a-z]+|[A-Z][a-z]+)"
CAPS = rf"{CAP}(?:[ ]{CAP}){{0,2}}"                  # one to three capitalised words, single spaces
# A title followed by either "." (with or without a space) or a space, so "Ms.North" and "Ms. North" both work
TITLE = r"(?:Mr|Mrs|Ms|Miss|Mx|Dr|Prof|Sir|Madam|Mme|Mlle)(?:\.[ ]*|[ ]+)"
GREET = r"(?:Dear|Hi|Hello|Hey|Hiya|Greetings|Welcome)"
SIGN = r"(?:Best regards|Kind regards|Regards|Sincerely|Cheers|Thanks|Thank you|Yours truly)"
ROLE = (r"(?:patient|customer|client|student|member|employee|applicant|account holder|user|resident|"
        r"guest|candidate|contact|attendee|nurse|doctor|coach|teacher|professor|mentor)")
# Introductions such as "I'm Garrick Murray", "my name is ...", "this is ..."
INTRO = (r"(?:name is|named|called|under the name(?: of)?|on behalf of|signed by|attn:?|attention:?|"
         r"I'm|I am|my name is|this is|speaking with)")
# Words after which a single known first name is likely a person ("reminder for Stephen", "Connect with Harvey")
FIRST_CTX = r"(?:for|with|to|ask|contact|call|from|by|cc|meet|tell|remind)"

# ---------- Arabic building blocks ----------
# Arabic letters plus diacritics (shadda, fatha, damma and so on), so words like حقّي stay in one piece
AR = r"\u0621-\u065F\u0670"
AR_WORD = rf"[{AR}]+"
AR_CUES = ("العميلة|العميل|السيدة|السيد|الآنسة|الأستاذة|الأستاذ|الدكتورة|الدكتور|المهندسة|المهندس|الشيخ|"
           "المريضة|المريض|الطالبة|الطالب|الموظفة|الموظف|المدعوة|المدعو|باسم|اسمها|اسمه|الاسم|اسم")

# Each rule: (kind, compiled regex, extra)
#   "cue"   rules mask capture group 1 (trimmed at the first stop word); extra = stop-word set
#   "pair"  rules mask two adjacent words if the first is a known first name and the second a known
#           last OR first name; extra = (first-name set, last-name set)
#   "first" rules mask a known first name (group 1) that follows a context word, plus an optional
#           second word (group 2) if it is also a known name; extra = (first-name set, last-name set)
RULES = [
    ("cue", re.compile(rf"(?<![A-Za-z]){TITLE}({CAPS})"), NOT_NAMES),
    ("cue", re.compile(rf"(?<![A-Za-z]){GREET}[,]?[ ]+({CAPS})"), NOT_NAMES),
    ("cue", re.compile(rf"(?<![A-Za-z]){SIGN}[,!.]?\s{{1,3}}({CAPS})"), NOT_NAMES),
    ("cue", re.compile(rf"(?<![A-Za-z])(?i:{ROLE})[ ]+({CAPS})"), NOT_NAMES),
    ("cue", re.compile(rf"(?<![A-Za-z])(?i:{INTRO})[ ]+({CAPS})"), NOT_NAMES),
    # Arabic cue word followed by up to six Arabic words (names with bin / ibn / al- chains)
    ("cue", re.compile(rf"(?<![{AR}])(?:{AR_CUES})(?![{AR}])[ ]*:?[ ]*({AR_WORD}(?:[ ]{AR_WORD}){{0,5}})"), AR_STOP),
    # a name at the very start of a text or line, followed by a comma and a typical opening word,
    # or by a request word, e.g. "Lincoln, We need..." or "Kattie could you..." (no name list needed)
    ("cue", re.compile(rf"(?:\A|(?<=\n))({CAP})(?=,[ ](?i:our|your|the|we|i|please|this|here|my|it)\b|[ ](?:could|can|would|will|please)\b)"), NOT_NAMES),
    ("pair", re.compile(rf"(?<![A-Za-z])({CAP})[ ]({CAP})(?![A-Za-z])"), (FIRST_EN, LAST_EN)),
    ("pair", re.compile(rf"(?<![{AR}])({AR_WORD})[ ]({AR_WORD})(?![{AR}])"), (FIRST_AR, LAST_AR)),
    # first name after a context word, e.g. "reminder for Stephen", "Connect with Harvey"
    ("first", re.compile(rf"(?<![A-Za-z])(?i:{FIRST_CTX})[ ]+({CAP})(?:[ ]({CAP}))?(?![A-Za-z])"), (FIRST_EN, LAST_EN)),
    # first name at the start of a sentence followed by a comma or a request word, e.g. "Kattie could you ..."
    ("first", re.compile(rf"(?:\A|(?<=[.!?][ ])|(?<=\n))({CAP})(?=,|[ ](?:could|can|would|will|please|we|I|you|thanks)\b)"
                         rf"(?:[ ]({CAP}))?"), (FIRST_EN, LAST_EN)),
    # a known first name after a comma at the end of a sentence, e.g. "Just a quick tip, Natasha."
    ("first", re.compile(rf"(?<=,[ ])({CAP})(?=[.!?](?:\s|$))"), (FIRST_EN, LAST_EN)),
]

def _trim(group_text, start, stop_words):
    """Keep the leading words of the group up to the first stop word; return (start, end) or None."""
    end = None
    for m in re.finditer(r"\S+", group_text):
        if m.group(0).strip(".,;:!?،؛").lower() in stop_words:
            break                                    # a stop word ends the name
        end = m.end()
    return None if end is None else (start, start + end)

def find_names(text):
    """Return a sorted list of (start, end, 'NAME') spans for person names."""
    claimed, spans = [], []
    for kind, pattern, extra in RULES:
        for m in pattern.finditer(text):
            if kind == "cue":
                trimmed = _trim(m.group(1), m.start(1), extra)
                if trimmed is None:
                    continue
                s, e = trimmed
            elif kind == "pair":                     # both words must be in the name lists
                first, last = extra
                # the second word may be a last name or a first name ("Cassie August")
                if m.group(1) not in first or (m.group(2) not in last and m.group(2) not in first) \
                        or m.group(1).lower() in NOT_NAMES:
                    continue
                s, e = m.start(1), m.end(2)
            else:                                    # "first": the first word must be a known first name
                first, last = extra
                if m.group(1) not in first or m.group(1).lower() in NOT_NAMES:
                    continue
                s, e = m.start(1), m.end(1)
                second = m.group(2) if m.lastindex and m.lastindex >= 2 else None
                if second and (second in last or second in first) and second.lower() not in NOT_NAMES:
                    e = m.end(2)                     # also take the second word when it is a known name
            if any(s < ce and e > cs for cs, ce in claimed):    # overlaps an earlier finding: skip
                continue
            claimed.append((s, e))
            spans.append((s, e, "NAME"))
    return sorted(spans)

Overwriting /home/linux1/bank-rag-guard/src/person_names.py


## 4. Combine structured identifiers and names (`src/masker.py`)
`find_all` runs the frozen structured masker first, then adds name findings that do not overlap them, so an identifier is never split by a name tag. `mask_all` replaces every finding with a tag such as `[IBAN]` or `[NAME]`.

In [4]:
%%writefile /home/linux1/bank-rag-guard/src/masker.py
"""Combined masker: structured identifiers first (pii.py), then person names (person_names.py)."""
import pii
import person_names

def find_all(text):
    """All PII spans as (start, end, label); name spans never overlap a structured finding."""
    spans = list(pii.find_pii(text))
    for s, e, label in person_names.find_names(text):
        if not any(s < ce and e > cs for cs, ce, _ in spans):
            spans.append((s, e, label))
    return sorted(spans)

def mask_all(text):
    """Return (masked_text, spans), with each finding replaced by a [LABEL] tag."""
    spans = find_all(text)
    pieces, last = [], 0
    for s, e, label in spans:
        pieces.append(text[last:s])                  # keep the text before the finding
        pieces.append(f"[{label}]")                  # replace the finding with its tag
        last = e
    pieces.append(text[last:])                       # keep the rest
    return "".join(pieces), spans

Overwriting /home/linux1/bank-rag-guard/src/masker.py


## 5. Try the name masker
Loads the modules and masks six hand-written sentences. **Expected:** names become `[NAME]` while titles stay visible; the Emirates ID and IBAN still become their own tags; the last sentence ("Customer Service") stays unchanged. The list sizes show how much name data was found; if they are 0, Faker could not be read.

In [5]:
import pii, person_names, masker
for module in (pii, person_names, masker):           # reload so edits to the files take effect
    importlib.reload(module)

print("Name list sizes: English first/last =", len(person_names.FIRST_EN), "/", len(person_names.LAST_EN),
      "| Arabic first/last =", len(person_names.FIRST_AR), "/", len(person_names.LAST_AR), "\n")

samples = [
    "Dear Hosea, please confirm that Mr. Ward and patient Leuschke attended. Regards, Agustin Perez",
    "Customer Allison Hill, Emirates ID 784-1958-1719583-7, reported a disputed card transaction.",
    "My name is Noah Rhodes and my IBAN is AE646921566149281915956.",
    "العميل محمود الألجاوي، رقم الهوية الإماراتية 784-2001-0109031-2، قدّم شكوى.",
    "تواصل السيد أحمد الفارسي مع الفرع بخصوص الحساب.",
    "The meeting with Customer Service is on Monday.",       # should stay unchanged
]
for text in samples:
    masked, spans = masker.mask_all(text)
    print("BEFORE:", text)
    print("AFTER :", masked, "\n")

Name list sizes: English first/last = 5038 / 8018 | Arabic first/last = 755 / 296 

BEFORE: Dear Hosea, please confirm that Mr. Ward and patient Leuschke attended. Regards, Agustin Perez
AFTER : Dear [NAME], please confirm that Mr. [NAME] and patient [NAME] attended. Regards, [NAME] 

BEFORE: Customer Allison Hill, Emirates ID 784-1958-1719583-7, reported a disputed card transaction.
AFTER : Customer [NAME], Emirates ID [EMIRATES_ID], reported a disputed card transaction. 

BEFORE: My name is Noah Rhodes and my IBAN is AE646921566149281915956.
AFTER : My name is [NAME] and my IBAN is [IBAN]. 

BEFORE: العميل محمود الألجاوي، رقم الهوية الإماراتية 784-2001-0109031-2، قدّم شكوى.
AFTER : العميل [NAME]، رقم الهوية الإماراتية [EMIRATES_ID]، قدّم شكوى. 

BEFORE: تواصل السيد أحمد الفارسي مع الفرع بخصوص الحساب.
AFTER : تواصل السيد [NAME] مع الفرع بخصوص الحساب. 

BEFORE: The meeting with Customer Service is on Monday.
AFTER : The meeting with Customer Service is on Monday. 



Checks that the new name rules are loaded, using sentences the old version handled wrongly.

In [6]:
# Confirm the NEW rules are loaded (the old file has no FIRST_CTX variable)
import person_names as pn
print("New rules loaded:", hasattr(pn, "FIRST_CTX"))   # must print True

# Sentences the old version got wrong; print the text each rule found
tests = [
    "Hello Mr.Kerluke, We received your last payment.",      # title with no space after the dot
    "I'm Garrick Murray. I will be picking up the results.",  # introduction cue, two-word name
    "Cardiac consult reminder for Stephen on Friday.",        # first name after 'for'
    "Dear Parents, our next field trip is organized.",        # group word: should stay unmasked
    "Hi, Music Club invites Ashly and friends.",              # group word: should stay unmasked
]
for t in tests:
    print([t[s:e] for s, e, _ in pn.find_names(t)], "<-", t)

New rules loaded: True
['Kerluke'] <- Hello Mr.Kerluke, We received your last payment.
['Garrick Murray'] <- I'm Garrick Murray. I will be picking up the results.
['Stephen'] <- Cardiac consult reminder for Stephen on Friday.
[] <- Dear Parents, our next field trip is organized.
[] <- Hi, Music Club invites Ashly and friends.


## 5b. Check the improved rules
Runs five short sentences through the name finder. **Expected:** `Kerluke` (title written without a space), `Garrick Murray` (introduction cue and a two-word name) and `Stephen` (first name after "for") are found; "Parents" and "Music Club" are left alone.

In [7]:
# Load the name finder by the same module name the rest of the notebook uses
import person_names as pn
importlib.reload(pn)                                   # re-read the file so the latest rules are used
print("New rules loaded:", hasattr(pn, "FIRST_CTX"))    # must print True

# Test sentences: the first three should each show a name; the last two should show nothing
tests = [
    "Hello Mr.Kerluke, We received your last payment.",       # title with no space after the dot
    "I'm Garrick Murray. I will be picking up the results.",  # introduction cue, two-word name
    "Cardiac consult reminder for Stephen on Friday.",        # first name after "for"
    "Dear Parents, our next field trip is organized.",        # group word: should stay unmasked
    "Hi, Music Club invites Ashly and friends.",              # group word: should stay unmasked
]
for t in tests:
    print([t[s:e] for s, e, _ in pn.find_names(t)], "<-", t)

New rules loaded: True
['Kerluke'] <- Hello Mr.Kerluke, We received your last payment.
['Garrick Murray'] <- I'm Garrick Murray. I will be picking up the results.
['Stephen'] <- Cardiac consult reminder for Stephen on Friday.
[] <- Dear Parents, our next field trip is organized.
[] <- Hi, Music Club invites Ashly and friends.


In [8]:
# Show where the notebook is running and which file each module was loaded from
import os, person_names
print("Notebook folder :", os.getcwd())
print("Module file     :", person_names.__file__)                 # must end in bank-rag-guard/src/person_names.py
print("New rules loaded:", hasattr(person_names, "FIRST_CTX"))     # must print True

Notebook folder : /home/linux1/bank-rag-guard
Module file     : /home/linux1/bank-rag-guard/src/person_names.py
New rules loaded: True


## 6. Score on the benchmark (English)
Compares our `[NAME]` findings with the benchmark's first, middle and last name labels:
- **Recall:** the share of labelled name words our masker covered (at least half of the word).
- **Precision (name label):** the share of our findings that overlap a labelled name.
- **Precision (any PII label):** the share that overlap any labelled PII, for example a username that contains a name.

Also lists examples of missed names and of findings that matched no label. These 1,000 examples are the ones we study while building the rules, so the final honest test will use a fresh slice.

In [9]:
NAME_GOLD = {"FIRSTNAME", "LASTNAME", "MIDDLENAME"}

def score_names(bench_df, finder):
    """Return (scores table, missed examples, unmatched findings) for the NAME tag."""
    gold_n = caught = pred_n = same = anyl = 0
    missed, wrong = [], []
    for row in bench_df.itertuples():
        text = row.source_text
        all_gold = get_mask(row.privacy_mask)
        gold = [g for g in all_gold if g["label"] in NAME_GOLD]
        preds = [p for p in finder(text) if p[2] == "NAME"]
        for g in gold:                                           # recall side
            gold_n += 1
            half = 0.5 * (g["end"] - g["start"])
            if any(overlap(p[0], p[1], g["start"], g["end"]) >= half for p in preds):
                caught += 1
            else:
                missed.append(("Missed", g["value"],
                               "…" + text[max(0, g["start"] - 35):g["start"]] + "[" + g["value"] + "]" +
                               text[g["end"]:g["end"] + 35] + "…"))
        for s, e, _ in preds:                                    # precision side
            pred_n += 1
            if any(overlap(s, e, g["start"], g["end"]) > 0 for g in gold):
                same += 1
            if any(overlap(s, e, g["start"], g["end"]) > 0 for g in all_gold):
                anyl += 1
            else:
                wrong.append(("Matched nothing", text[s:e],
                              "…" + text[max(0, s - 35):s] + "[" + text[s:e] + "]" + text[e:e + 35] + "…"))
    table = pd.DataFrame([{"Tag": "NAME", "Labelled name words": gold_n, "Recall": caught / max(gold_n, 1),
                           "Findings": pred_n, "Precision (name label)": same / max(pred_n, 1),
                           "Precision (any PII label)": anyl / max(pred_n, 1)}])
    return table, missed, wrong

name_scores, missed_names, wrong_names = score_names(bench, masker.find_all)
rates = ["Recall", "Precision (name label)", "Precision (any PII label)"]
show(name_scores, "Name masker on the ai4privacy benchmark (1,000 English examples)",
     pct_cols=rates, bars={c: 1 for c in rates})

examples = pd.DataFrame(missed_names[:12] + wrong_names[:8], columns=["Kind", "Name", "Context"])
show(examples, "Examples of missed names and findings that matched nothing")

**Name masker on the ai4privacy benchmark (1,000 English examples)**

Tag,Labelled name words,Recall,Findings,Precision (name label),Precision (any PII label)
NAME,462,68.4%,365,77.8%,100.0%


**Examples of missed names and findings that matched nothing**

Kind,Name,Context
Missed,Nels,"…""[Nels], the code you submitted for the ne…"
Missed,Garett,…Requested [Garett] to lead advanced diagnostic tool d…
Missed,Dejuan,…Billing details for [Dejuan] Mertz from Herminiastead need an u…
Missed,Mertz,…Billing details for Dejuan [Mertz] from Herminiastead need an update.…
Missed,Dashawn,"…Join the seminar with [Dashawn], a renowned leadership coach, on 1…"
Missed,Karolann,…Your subscription will expire soon [Karolann]. Do consider renewing using your 0…
Missed,Ashly,"…Hi, Music Club invites [Ashly] and friends to the music night. Em…"
Missed,Daija,"…stigma in our school, Eveline. Can [Daija] from McGlynn and Sons give a speec…"
Missed,Emelie,… wanted to confirm if this was you [Emelie].…
Missed,Dolores,…[Dolores] needs to be assessed for possible …


## 7. Customer notes and false alarms
- **Customer notes (synthetic, English and Arabic):** for each note, are the true name's words gone from the masked text? These notes were generated with the same Faker lists the masker uses, so the numbers are **optimistic**; they are a sanity check, not an independent score.
- **Enron emails:** the share of emails where `[NAME]` fires, plus 20 random matches with context. The emails are real, public messages, so most matches should be real people's names. Check that they look like genuine names rather than ordinary words, and hide the names before putting this table in a report.

In [10]:
# --- Customer notes: are the true name's words still visible after masking? ---
SKIP = {"Mr", "Mrs", "Ms", "Miss", "Dr", "Prof", "Jr", "Sr", "II", "III", "IV", "MD", "DDS", "DVM", "PhD"}

def name_words(full_name):
    """The words of a name that must be hidden (titles and suffixes are allowed to stay visible)."""
    return [w for w in full_name.replace(",", " ").split() if w.strip(".") not in SKIP and len(w) > 1]

rows = []
for r in customers.itertuples():
    masked, _ = masker.mask_all(r.text)
    words = name_words(r.name)
    visible = [w for w in words if w in masked]                  # name words that survived masking
    rows.append({"Language": "Arabic" if r.language == "ar" else "English",
                 "All words masked": not visible,
                 "Share of words masked": 1 - len(visible) / max(len(words), 1)})
res = pd.DataFrame(rows)
summary = res.groupby("Language").agg(Notes=("All words masked", "size"),
                                      Fully_masked=("All words masked", "mean"),
                                      Words_masked=("Share of words masked", "mean")).reset_index()
show(summary.rename(columns={"Fully_masked": "Notes fully masked", "Words_masked": "Share of name words masked"}),
     "Customer notes: name masking (synthetic data, optimistic)",
     pct_cols=["Notes fully masked", "Share of name words masked"])

# --- Enron: how often does [NAME] fire, and are the matches real names? ---
pool, emails_hit = [], 0
for text in enron["text"]:
    spans = [s for s in masker.find_all(text) if s[2] == "NAME"]
    emails_hit += bool(spans)
    for s, e, _ in spans:
        pool.append(("…" + text[max(0, s - 35):s].replace("\n", " ") + "[" + text[s:e] + "]" +
                     text[e:e + 35].replace("\n", " ") + "…"))
display(Markdown(f"**[NAME] fired in {emails_hit} of {len(enron):,} emails ({emails_hit / len(enron):.1%}); "
                 f"{len(pool):,} matches in total. Random sample of 20:**"))
sample = random.Random(11).sample(pool, min(20, len(pool)))
show(pd.DataFrame({"Match in context": sample}), "Random Enron matches")

**Customer notes: name masking (synthetic data, optimistic)**

Language,Notes,Notes fully masked,Share of name words masked
Arabic,152,98.7%,99.8%
English,148,99.3%,99.7%


**[NAME] fired in 1032 of 1,500 emails (68.8%); 4,125 matches in total. Random sample of 20:**

**Random Enron matches**

Match in context
…; Jacqueline Java; Kimberly Curry; [Paul Fox]; Tracey Bradley; gfergus@brobeck.c…
"…hanks, April Williams Assistant to [Paul] Keeler & Lori Marlatt ---------- O…"
…rketers in equal shares. Thanks. [Ray] -----Original Message----- From: …
"…fore it goes back to OXY. Thanks, [Rich]. <>…"
…with Pinnacle on the 19th? Thanks [Amy]…
"… one of these days. Regards, [Nasim Khan]…"
…--------------------- Forwarded by [Greg Wolfe]/HOU/ECT on 04/16/2001 07:26 AM ---…
"…ave any questions, please contact [Jackie Gallagher] at 202-628-8200 or jgallagher@eps…"
…Subject: Notice of Customer [Registration] at Business Week Online To: doug …
…oved. An after the fact note from [Linda] will suffice. Dan ---------------…


## 8. Arabic notes where a name word is still visible
Lists the Arabic customer notes in which at least one word of the true name remains after masking, together with the visible words and the masked text. This shows which Arabic name patterns the rules miss.

In [11]:
# For every Arabic note, find name words that are still visible after masking
failed = []
for r in customers[customers["language"] == "ar"].itertuples():
    masked, _ = masker.mask_all(r.text)
    still_visible = [w for w in name_words(r.name) if w in masked]    # name words that survived masking
    if still_visible:
        failed.append({"True name": r.name,
                       "Visible words": " | ".join(still_visible),
                       "Masked text": masked})

print(len(failed), "Arabic notes with a visible name word")
show(pd.DataFrame(failed[:15]), "First 15 failing Arabic notes")      # send me this table

2 Arabic notes with a visible name word


**First 15 failing Arabic notes**

True name,Visible words,Masked text
السيد مرتضي الحجر بن الهنوء بن الأزد,الأزد,العميل [NAME] الأزد، رقم الهوية الإماراتية [EMIRATES_ID]، رقم الآيبان [IBAN]، الهاتف [PHONE]، قدّم شكوى بخصوص عملية بطاقة مشكوك فيها.
المهندسة أغاريد الحجر بن الهنوء بن الأزد,الأزد,العميل [NAME] الأزد، رقم الهوية الإماراتية [EMIRATES_ID]، رقم الآيبان [IBAN]، الهاتف [PHONE]، قدّم شكوى بخصوص عملية بطاقة مشكوك فيها.


## 9. Honest test on a fresh slice
Downloads 1,000 benchmark examples from a part of the dataset we have never looked at (rows starting at 60,000), keeps only the same language as our development set, saves them to `data/pii_benchmark_fresh.jsonl`, and scores the frozen name masker on them. Because the rules were not tuned on these examples, this is the number to report.

In [14]:
import requests, time

API = "https://datasets-server.huggingface.co/rows"
DATASET = "ai4privacy/pii-masking-200k"
LANG = bench["language"].iloc[0]                    # same language as the development set ('en')
FRESH_PATH = f"{DATA_DIR}/pii_benchmark_fresh.jsonl"

def fetch_page(offset, length=100):
    """Fetch one page of rows from the Hugging Face datasets server, retrying on temporary errors."""
    for attempt in range(6):
        r = requests.get(API, params={"dataset": DATASET, "config": "default", "split": "train",
                                      "offset": offset, "length": length}, timeout=60)
        if r.status_code == 200:
            return [x["row"] for x in r.json()["rows"]]
        if r.status_code in (429, 500, 502, 503):   # temporary: wait and try again
            time.sleep(5 * 2 ** attempt)
            continue
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:200]}")   # anything else: stop and show it
    raise RuntimeError("Too many retries")

if os.path.exists(FRESH_PATH) and os.path.getsize(FRESH_PATH) > 0:
    fresh = pd.read_json(FRESH_PATH, lines=True, dtype=False, convert_dates=False)   # reuse the saved slice
else:
    # Step 1: probe the dataset to see which rows are in our language (the data is sorted by language)
    probes = {}
    for off in range(5000, 60001, 5000):
        probes[off] = fetch_page(off, 1)[0]["language"]
    print("Language at each probed offset:", probes)

    # Step 2: start from the first English probe that is not the 20,000 area used earlier
    english_starts = [o for o, lang in probes.items() if lang == LANG and o != 20000]
    if not english_starts:
        raise RuntimeError(f"No '{LANG}' rows found at the probed offsets; send me the line printed above.")
    START = english_starts[0]
    print("Collecting from offset", START)

    # Step 3: page forward, keeping only our language and never an example from the development set
    rows, offset, empty_pages = [], START, 0
    seen_ids = set(bench["id"].astype(str))
    while len(rows) < 1000 and empty_pages < 3 and offset < START + 8000:
        got = [r for r in fetch_page(offset) if r["language"] == LANG and str(r["id"]) not in seen_ids]
        rows.extend(got)
        empty_pages = empty_pages + 1 if not got else 0   # stop after 3 pages with no English rows
        offset += 100
        print(f"  offset {offset}: {len(rows)} examples so far")   # progress
    if not rows:
        raise RuntimeError("No rows collected; send me the output above.")
    fresh = pd.DataFrame(rows[:1000])
    fresh.to_json(FRESH_PATH, orient="records", lines=True, force_ascii=False)

print(len(fresh), "fresh examples; language:", LANG, "; overlap with development ids:",
      len(set(fresh["id"].astype(str)) & set(bench["id"].astype(str))))

# Score the frozen name masker on the fresh slice
fresh_scores, fresh_missed, fresh_wrong = score_names(fresh, masker.find_all)
rates = ["Recall", "Precision (name label)", "Precision (any PII label)"]
show(fresh_scores, "Name masker on a fresh, unseen slice (honest score)", pct_cols=rates, bars={c: 1 for c in rates})
show(pd.DataFrame(fresh_missed[:12] + fresh_wrong[:8], columns=["Kind", "Name", "Context"]),
     "Examples of missed names and findings that matched nothing (fresh slice)")

Language at each probed offset: {5000: 'en', 10000: 'en', 15000: 'en', 20000: 'en', 25000: 'en', 30000: 'en', 35000: 'en', 40000: 'en', 45000: 'fr', 50000: 'fr', 55000: 'fr', 60000: 'fr'}
  offset 5100: 100 examples so far
  offset 5200: 200 examples so far
  offset 5300: 300 examples so far
  offset 5400: 400 examples so far
  offset 5500: 500 examples so far
  offset 5600: 600 examples so far
  offset 5700: 700 examples so far
  offset 5800: 800 examples so far
  offset 5900: 900 examples so far
  offset 6000: 1000 examples so far
1000 fresh examples; language: en ; overlap with development ids: 0


**Name masker on a fresh, unseen slice (honest score)**

Tag,Labelled name words,Recall,Findings,Precision (name label),Precision (any PII label)
NAME,485,67.0%,348,80.2%,98.6%


**Examples of missed names and findings that matched nothing (fresh slice)**

Kind,Name,Context
Missed,Devante,"…re thrilled to welcome you aboard, [Devante]. You can access your assigned work…"
Missed,Erling,"…[Erling], there's been an unidentified sign…"
Missed,O'Conner,"…Dear parents of [O'Conner], The school fees for distance lear…"
Missed,Lou,"…et at 562, Apt. 691, West Michele. [Lou] is the point of contact at +71 064…"
Missed,Derick,"…lth and wellbeing for members like [Derick] who is 48 years years old, a criti…"
Missed,Augustus,…[Augustus] made some connections between her …
Missed,Roberts-Rolfson,"…For [Roberts-Rolfson] and team, the weekly health care l…"
Missed,Bennett,…The meeting with [Bennett] will be conducted at the county of…
Missed,Adonis,"…Green Capsule' marketing campaign. [Adonis], can you confirm?…"
Missed,Zelma,"… into the insurance claim filed by [Zelma], regarding vehicle AE12TZH involve…"


In [13]:
# Remove the empty file saved by the failed run so it is not reused
if os.path.exists(FRESH_PATH):
    os.remove(FRESH_PATH)
    print("Deleted", FRESH_PATH)

# Look at one page from the server: its columns, the language values, and our development language
page = fetch_page(START, 100)
print("Rows returned:", len(page))
if page:
    print("Columns:", list(page[0].keys()))
    print("Language values on this page:", sorted({str(r.get("language")) for r in page}))
print("Development language value (LANG):", repr(LANG))
print("Development 'set' values:", sorted(bench["set"].astype(str).unique()))

Deleted /home/linux1/bank-rag-guard/data/pii_benchmark_fresh.jsonl
Rows returned: 100
Columns: ['source_text', 'target_text', 'privacy_mask', 'span_labels', 'mbert_text_tokens', 'mbert_bio_labels', 'id', 'language', 'set']
Language values on this page: ['fr']
Development language value (LANG): 'en'
Development 'set' values: ['train']
